# Decision Tree Algorithm

## What is a Decision Tree?
A Decision Tree is a supervised machine learning algorithm used for both classification and regression tasks. It has a hierarchical, tree structure, which consists of a root node, branches, internal nodes and leaf nodes.
- **Root Node**: Represents the entire population or sample and this further gets divided into two or more homogeneous sets.
- **Internal Nodes**: Represents a test on an attribute (e.g., whether a coin flip comes up heads or tails).
- **Branches**: Represents the outcome of the test.
- **Leaf Nodes**: Represents the final decision or class label.

## Why is it used?
1. **Interpretability**: Decision trees create a model that predicts the value of a target variable by learning simple decision rules inferred from the data features. They are easy to understand and visualize.
2. **No Normalization Required**: Unlike distance-based algorithms (like K-Means or KNN), decision trees do not require feature scaling or normalization.
3. **Handles Non-Linear Relationships**: Decision trees can capture non-linear relationships between features and the target variable.
4. **Feature Importance**: They provide a clear indication of which features are most important for prediction.

## Code Explanation

This implementation uses the **ID3 (Iterative Dichotomiser 3)** algorithm to build the decision tree. Here is the significance of each function:

1. **`calculate_entropy(labels)`**:
    - **Purpose**: Computes the entropy (impurity) of a set of examples.
    - **Significance**: Entropy measures the disorder or uncertainty in the data. Lower entropy means the data is purer (mostly one class), while higher entropy means the data is mixed. It is crucial for determining how well a split separates the classes.

2. **`calculate_information_gain(examples, attr, target_attr)`**:
    - **Purpose**: Calculates the reduction in entropy achieved by partitioning the examples according to a given attribute.
    - **Significance**: This is the core metric used by ID3 to select the best attribute for splitting. The attribute with the highest information gain is chosen as the node for the tree at that step, as it provides the most useful information for classification.

3. **`majority_class(examples, target_attr)`**:
    - **Purpose**: Determines the most frequent class label in the given set of examples.
    - **Significance**: Used as a fallback mechanism. If there are no attributes left to split on, or if a branch has no examples, we default to the majority class to make a prediction. This prevents the tree from failing when data is inseparable or missing.

4. **`learn_decision_tree(examples, attributes, target_attr)`**:
    - **Purpose**: Recursively builds the decision tree structure.
    - **Significance**: This is the main driver function. It handles the base cases (all examples same class, or no attributes left) and the recursive step (selecting the best attribute, splitting data, and calling itself for subtrees). It constructs the final tree dictionary.

In [ ]:
import math
from collections import Counter

def calculate_entropy(labels: list) -> float:
    """Calculate the entropy of a list of labels."""
    if not labels:
        return 0.0
    counts = Counter(labels)
    total = len(labels)
    ent = 0.0
    for cls, cnt in counts.items():
        p = cnt / total
        ent += -p * math.log(p, 2)
    return ent


def calculate_information_gain(examples: list[dict], attr: str, target_attr: str) -> float:
    """Calculate the information gain of splitting on attr."""
    parent_labels = [ex[target_attr] for ex in examples]
    gain = calculate_entropy(parent_labels)

    total = len(examples)
    value_counts = Counter(ex[attr] for ex in examples)

    for v in sorted(value_counts.keys()):  # sorted for deterministic behavior
        subset_labels = [ex[target_attr] for ex in examples if ex[attr] == v]
        gain -= (value_counts[v] / total) * calculate_entropy(subset_labels)

    return gain


def majority_class(examples: list[dict], target_attr: str) -> str:
    """Return the majority class. Break ties alphabetically."""
    labels = [ex[target_attr] for ex in examples]
    counts = Counter(labels)

    max_count = max(counts.values())
    tied = [cls for cls, c in counts.items() if c == max_count]
    return min(tied)  # alphabetical tie-break


def learn_decision_tree(examples: list[dict], attributes: list[str], target_attr: str):
    """Build a decision tree using the ID3 algorithm (entropy + information gain)."""
    labels = [ex[target_attr] for ex in examples]

    # Base case 1: all same class
    if len(set(labels)) == 1:
        return labels[0]

    # Base case 2: no attributes left
    if not attributes:
        return majority_class(examples, target_attr)

    # Choose best attribute by IG (tie-break: first in attributes list)
    best_attr = attributes[0]
    best_gain = calculate_information_gain(examples, best_attr, target_attr)

    for attr in attributes[1:]:
        gain = calculate_information_gain(examples, attr, target_attr)
        if gain > best_gain:  # strict > keeps earlier attribute on ties
            best_gain = gain
            best_attr = attr

    tree = {best_attr: {}}

    # Branch in sorted order of values for consistent structure
    values = sorted(set(ex[best_attr] for ex in examples))
    remaining_attrs = [a for a in attributes if a != best_attr]

    for v in values:
        subset = [ex for ex in examples if ex[best_attr] == v]

        # Empty branch -> majority class of current examples
        if not subset:
            tree[best_attr][v] = majority_class(examples, target_attr)
        else:
            tree[best_attr][v] = learn_decision_tree(subset, remaining_attrs, target_attr)

    return tree